In [1]:
from cryptography.hazmat.primitives import hashes

import cryptography.hazmat.primitives.kdf.argon2
import os


bytearray(b'')


In [4]:
from cryptography.hazmat.primitives.kdf.argon2 import Argon2id

password = b"hello"
salt1 = os.urandom(16)
kdf1 = Argon2id(salt1, 16, 2, 4, 1024*64)
hidden = kdf1.derive(password)
print(hidden.hex())







dbe0eb170d136a6eb3115586ab7be107


In [47]:
import os
from cryptography.hazmat.primitives import hashes

from cryptography.hazmat.primitives.kdf.argon2 import Argon2id

class MfaKdf:
    def __init__(self, iterations = 2, lanes = 4, memory = 64*1024, existing_salt = None):
        self.iterations = iterations
        self.lanes = lanes
        self.memory = memory
        self._used = False
        if existing_salt is not None:
            self.salt = existing_salt
        else:
            self.salt = os.urandom(16)
        
    @property
    def salt (self):
        return self._salt
    
    @salt.setter
    def salt(self, value):
        if isinstance(value, bytes):
            if len(value) == 16:
                self._salt = value
            else:
                raise ValueError("Salt must be 16 bytes")
        else:
            raise TypeError("Invalid salt type")
    
    def derive(self,  master_password, weak_password):
        if not self._used:    
            master_password = master_password.encode('utf-8') 
            weak_password = weak_password.encode('utf-8') 
            kdf1 = Argon2id(self._salt, 16, self.iterations, self.lanes, self.memory)
            hidden_salt = kdf1.derive(weak_password)
            kdf2 = Argon2id(hidden_salt, 32, self.iterations, self.lanes, self.memory)
            self.hash = kdf2.derive(master_password)
            self._used = True
            master_password = b""
            weak_password = b""
            hidden_salt = b""
            return(self.hash)
        else: 
            raise Exception("Instance already used, cannot use more than once")
    def verify(self, master_password, weak_password, hash):
        if self._used is False:
            if self.derive(master_password, weak_password) == hash:
                return True
            else:
                return False
        
  

In [ ]:
import os
from abc import ABC, abstractmethod
from cryptography.hazmat.primitives.ciphers.aead import AESGCM
from cryptography.exceptions import InvalidTag

class SymmetricCipherEngine(ABC):
    def __init__(self, hashed_key: bytes):
        self.hashed_key = hashed_key
        self._chunk_size = 2*1024
        
    @property
    def hashed_key(self):
        return self._hashed_key
        
    @hashed_key.setter
    def hashed_key(self, value):
        if not isinstance(value, bytes):
            raise TypeError("The hashed key is not binary")
        if len(value) != 32:
            raise ValueError("The hashed key not 32 bytes")
        self._hashed_key = value        
        
    @abstractmethod
    def encrypt_file(self):
        pass
    @abstractmethod
    def decrypt_file(self):
        pass

class AesGcm(SymmetricCipherEngine):
    def __init__(self, hashed_key: bytes):
        super().__init__(hashed_key)
        self._cipher = AESGCM(self._hashed_key)
        self._used = False

    def encrypt_file(self, input_file: str, vault_file: str):
        file_name = os.path.basename(input_file)
        meta_data = []

        if self._used == False:
            with open(input_file, 'rb') as infile, open(vault_file, 'wb') as vfile:
                while True:
                    data = infile.read(self._chunk_size) 
                    if not data:
                        break 
                    nonce = os.urandom(12)
                    ciphertext = self._cipher.encrypt(nonce, data, None)
                    start_ctr = vfile.tell()
                    vfile.write(ciphertext)
                    end_ctr = vfile.tell()

                    meta_data.append({
                        "IV": nonce,
                        "Start" : start_ctr,
                        "End" : end_ctr
                    })
            self._used = True
            print(f"File Sucessfully encrypted to {vault_file}")
            return meta_data


    def decrypt_file(self, meta_data : list, scaf_file : str, output_file : str):
        if self._used == False:
            self._used = True
            with open(scaf_file, 'rb') as vfile, open(output_file, 'ab') as outfile:
                for chunk in meta_data:
                    cr_iv = chunk["IV"]
                    vfile.seek(chunk["Start"])
                    data_len = chunk["End"] - chunk["Start"]
                    read_cipher = vfile.read(data_len)
                    try:
                        plaintxt = self._cipher.decrypt(cr_iv, read_cipher, None)
                        outfile.write(plaintxt)
                    except InvalidTag:
                        print("File Authentication failed! File has been corrupted/tampered")
                        return False
                return True
        else: 
            print("Instance already used, cannot use more than once")
            return False

                



In [66]:
salt = os.urandom(16)
Kdf = MfaKdf(existing_salt = salt)
hash = Kdf.derive("b1d2sxmbek", "varun17")
vault1 = AesGcm(hash)
hash = b""
meta_data = vault1.encrypt_file(r"C:\Users\varun\Project - SecureVault\venv\modules\NIST-Paper.pdf", r"C:\Users\varun\Project - SecureVault\venv\modules\NIST-Paper.scaf")



File Sucessfully encrypted to C:\Users\varun\Project - SecureVault\venv\modules\NIST-Paper.scaf


In [67]:
hash1 = MfaKdf(existing_salt = salt).derive("b1d2sxmbek", "varun17")
vault2 = AesGcm(hash1)
hash1 = b""
vault2.decrypt_file(meta_data, r"C:\Users\varun\Project - SecureVault\venv\modules\NIST-Paper.scaf", r"C:\Users\varun\Project - SecureVault\venv\modules\NIST-Paper.pdf" )


True

In [4]:
import time
import msgpack
import sqlite3

class VaultMap:
    def __init__(self, existing_map : bytes = None):
        if existing_map:
            self.map = msgpack.unpackb(existing_map, strict_map_key = False)
        else:
            self.map = {}
    
    def add_file(self,virtual_path : str, chunk_data : list, file_size: int):
        self.map[virtual_path] = {
            "is_active" : True,     #setting true by default, since new file added, if wish to set it false use the method delete file
            "created_at" : int(time.time()),
            "total_size" : file_size,
            "meta_data" : chunk_data
        }
    
    def set_active_status(self, virtual_path : str, status: bool):
        if virtual_path in self.map:
            self.map[virtual_path]["is_active"] = status
        else:
            raise KeyError("Path does not exist in vault map")
        
    def pack_map(self):
        pkb = msgpack.packb(self.map, use_bin_type = True)
        return pkb

    def get_meta(self, virtual_path : str):
        if virtual_path in self.map:
            return self.map.get(virtual_path)
        
class VaultDatabase:
    def __init__(self):
        self.db = sqlite3.connect(':memory:')
        self.cursor = self.db.cursor()
        self.db._create_schema()

    def db_map_sync(self, vault_map_dict: dict):
        info = []
        for virtual_path, meta_data in vault_map_dict:
            file_name = os.path.basename(virtual_path)
            relative_dir = os.path.dirname(virtual_path) + '/'
            info.append((
                file_name,
                relative_dir,
                virtual_path,
                meta_data['total_size'],
                meta_data['created_at'],
                meta_data['is_active']
            ))
        self.cursor.execute('INSERTMANY INTO files VAULES (?,?,?,?,?,?)', info)
        self.db.commit()


    def _create_schema(self):
        self.cursor.execute('''
            CREATE TABLE IF NOT EXISTS files (
                id INTEGER PRIMARY KEY AUTOINCREMENT,
                filename TEXT NOT NULL,
                parent_folder TEXT NOT NULL,
                full_path TEXT UNIQUE NOT NULL,
                size_bytes INTEGER,
                created_at INTEGER,
                is_active BOOLEAN
            )
        ''')
        
        self.cursor.execute('CREATE INDEX idx_folder ON files(parent_folder)')
        self.db.commit()
    
    def inmo_file(self, virtual_path: str, meta_dict: dict):
        '''Insert or Modify file metadata'''
        file_name = os.path.basename(virtual_path)
        relative_dir = os.path.dirname(virtual_path) + '/'
        params = (file_name, relative_dir, virtual_path,
                meta_data['total_size'],
                meta_data['created_at'],
                meta_data['is_active'])

        query = 'INSERT OR REPLACE INTO files VAULES (?,?,?,?,?,?)'
        self.cursor.execute(query, params)

    def get_files_in_folder(self, folder_path: str):
        if not folder_path.endswith('/'):
            folder_path += '/'
        query = 'SELECT file_name, size_bytes, created_at FROM files WHERE relative_dir = ? AND is_active = 1'
        self.cursor.execute(query, (folder_path,))
        return self.cursor.fetchall()


In [ ]:

import struct
import os 

class VaultIO:
    def __init__(self, filepath : str):
        self.filepath = filepath
        
    
    @staticmethod
    def conditional_format(input_str, format_char):
        return [int(part) for part in input_str.split(format_char)]
    
    def initialize_new_vault(self, version, salt, pointer_iv, pointer_ciphertext):
        major, minor, patch = self.conditional_format(version, ".")
        header_format = '>4sBBB16s12s32s'
        cipher_stream = struct.pack(
            header_format,
            b"SCAF",
            major,
            minor,
            patch,
            salt,
            pointer_iv,
            pointer_ciphertext
        )

        with open(self.filepath, 'wb') as nfile:
            nfile.write(cipher_stream)

    def read_header(self) -> tuple:
        header_format = '>4sBBB16s'
        with open(self.filepath, 'rb') as rfile:
            header = rfile.read(23)  
            output = struct.unpack(header_format, header)
        return output  

    def read_pointer(self) -> tuple:
        header_format = '>12s32s'
        with open(self.filepath, 'rb') as rfile:
            rfile.seek(23)
            pointer = rfile.read(44)
            output = struct.unpack(header_format, pointer)
        return output  
    
    def append_cargo(self, encrypted_chunk : bytes):
        with open(self.filepath, 'ab') as afile:
            afile.write(encrypted_chunk) #assuming that while editing, cargo added first, hash appended later

    def read_cargo(self, start_byte, end_byte):
        byte_len = end_byte - start_byte
        with open(self.filepath, 'rb') as rfile:
            rfile.seek(start_byte)
            stream = rfile.read(byte_len)
        return stream
    
    def read_map(self, pointer_offset):
        with open(self.filepath, 'rb') as rfile:
            rfile.seek(pointer_offset)
            stream = rfile.read()
        return stream
    
    def atomic_map_swap(self, map_pointer, nmap_pointer_iv, nmap_pointer_cipher, nmap_pointer, nmap_iv, nmap_cipher):
        backup_path = self.filepath + ".bak"
        with open(self.filepath, 'r+b') as f:
            f.seek(23)
            map_pointer_iv = f.read(12)
            map_pointer_cipher = f.read(32)
            f.seek(map_pointer)
            map_iv = f.read(12)
            map_cipher = f.read()
        
        n = len(map_cipher)
        backup_format = f'>12s32s12s{n}s'
        backup_stream = struct.pack(backup_format,
                                map_pointer_iv,
                                map_pointer_cipher,
                                map_iv,
                                map_cipher
                                )

        with open(backup_path, 'wb') as bak:
            bak.write(backup_stream)
            os.fsync(bak.fileno())
    
        with open(self.filepath, 'r+b') as f:
            f.seek(nmap_pointer)
            f.write(nmap_iv)
            f.write(nmap_cipher)
            f.seek(23)
            f.write(nmap_pointer_iv)
            f.write(nmap_pointer_cipher)
            os.fsync(f.fileno())


In [ ]:
import tempfile
import secrets
import os
from pathlib import Path
import sys
import subprocess

class GhostFileIO:
    def __init__(self):
        self._tempdir_obj = tempfile.TemporaryDirectory()
        self.tempdir = Path(self._tempdir_obj.name)
        self.writechunk = 2*1024*1024  
    
    def create_ghost_path(self, filename):
        basename = secrets.token_hex(16)
        filebase, filebase_extension = os.path.splitext(filename)
        ghost_filepath = os.path.join(self.tempdir, f"{basename}{filebase_extension}")
        return str(ghost_filepath)
    
    def write_decrypted_chunk(self, path, chunk):
        with open(path, 'ab') as tmp:
            tmp.write(chunk)


    def secure_shred(self, path):
        if not os.path.exists(path):
            return
        else:
            filesize = os.path.getsize(path)
            with open(path, 'r+b') as tmp:
                for i in range(0, filesize, self.writechunk):
                    bytes_to_write  = min(self.writechunk, filesize - tmp.tell())
                    tmp.write(b'\x00' * bytes_to_write)
                os.fsync(tmp.fileno())
            os.remove(path)

    def wipe_session(self):
        for path in self.tempdir.rglob('*'):
            if path.is_file():
                self.secure_shred(path)
        self._tempdir_obj.cleanup()
    
    def launch_file(self, path):
        if sys.platform == "win32": #for windows
            os.startfile(path)
        elif sys.platform == "darwin": # macOS
            subprocess.call(["open", path])
        else: 
            subprocess.call(["xdg-open", path]) #for linux

23

In [ ]:
import concurrent.futures
from cryptography.exceptions import InvalidTag
import time

from modules.crypto_core import MfaKdf, AesGcm
from modules.vaultmap import hashmap, sqlitemap
from modules.system import VaultIO, GhostFileIO

class VaultManager:
    def __init__(self, filepath):
        self.filepath = filepath
        self.vaultio = VaultIO(self.filepath)
        
        
        
    
    def unlock_hash(self, master_password, weak_password):
        self.check, self.major, self.minor, self.patch, self.salt = self.vaultio.read_header()
        kdf = MfaKdf(existing_salt = self.salt)
        self.hash = kdf.derive(master_password, weak_password)
    #Category 1 : Lifecycle and Authentication
    def boot_vault(self):
        self.aesgcm = AesGcm(self.hash)
        if os.path.exists(self.filepath):
            iv_identifier, identifier_ct = self.vaultio.read_pointer()
            identifier_pt = self.aesgcm.decrypt_cipherstream(iv_identifier, identifier_ct)
            P, L = struct.unpack(">QQ", identifier_pt)
            map_cipher = self.vaultio.read_map(P,L)
            map_iv = map_cipher[:12]
            map_cipher = map_cipher[12:]
            map_pt = self.aesgcm.decrypt_cipherstream(map_iv, map_cipher)
            
            
            self.hashmap = VaultMap(map_pt)
            del map_pt
            self.vaultdb = VaultDatabase()
            self.vaultdb.db_map_sync(self.hashmap.map)
            for path, pathinfo in self.hashmap.map.items():
                self.hashmap.map[path] = pathinfo["meta_data"]
        else:
            salt = os.urandom(16)
            map_identifier_iv = os.urandom(12)
            P, L = 67, 0
            map_identifier_pt = struct.pack('>QQ', P, L)
            map_identifier_ct = self.aesgcm.encrypt_cipherstream(map_identifier_pt, map_identifier_iv)
            self.vaultio.initialize_new_vault("1.0.0.", salt, map_identifier_iv, map_identifier_ct)
            salt, map_identifier_iv, map_identifier_ct = b"", b"", b""
            self.hashmap = VaultMap()
            self.vaultdb = VaultDatabase()
            self.vaultdb._create_schema()


    def lock_vault(self):
        if hasattr(self, 'hashmap') and self.hashmap is not None:
            self.sync_map_to_disc()
            self.hashmap.destroy()
            del self.hashmap
        
        if hasattr(self, 'vaultdb') and self.vaultdb is not None:
            self.vaultdb.close_connection()
            del self.vaultdb
            
        if hasattr(self, 'hash'):
            self.hash = b'\x00' * len(self.hash) 
            del self.hash
            
        if hasattr(self, 'aesgcm'):
            del self.aesgcm

    #Category 2 : Ingestion

    def _chunked_coordinates(self, filepath :str, chunk_io : int = 2*1024*1024):
        filesize = os.path.getsize(filepath)
        for offset in range(0, filesize, chunk_io):  #start with 0 read 0 then read chunk_io 
            size = min(offset, filesize - offset)
            yield (filepath, offset, size)

    
    def _worker_encrypt(self, chunked_coordinates : tuple):
        chunk_io = 2*1024*1024
        filepath, offset, size = chunked_coordinates
        self.aesgcm = AesGcm(self.hash)
        with open(filepath, 'rb') as f:
            pt_chunk = f.read(size)
        iv, ciphertext = self.aesgcm.encrypt_cipherstream(pt_chunk)
        return iv, ciphertext
    
    def sync_map_to_disc(self):
        nmap = self.vaultdb.get_complete_meta()
        nmap_pt = self.hashmap.restitch(nmap)
        nmap_iv, nmap_ciphertext = self.aesgcm.encrypt_cipherstream(nmap_pt)
        nmap_P = self.vaultio.get_eof()
        nmap_L =  len(nmap_iv) + len(nmap_ciphertext)
        identifier_pt = struct.pack('>QQ', nmap_P, nmap_L)
        iv_identifier, identifier_ct = self.aesgcm.encrypt_cipherstream(identifier_pt)
        map_pointer_iv, map_pointer_cipher = self.vaultio.read_pointer()
        map_pointer_pt = self.aesgcm.decrypt_cipherstream(map_pointer_iv, map_pointer_cipher)
        map_P, map_L = struct.unpack('>QQ', map_pointer_pt)
        self.vaultio.atomic_map_swap(map_P,map_L, iv_identifier, identifier_ct, nmap_P, nmap_iv, nmap_ciphertext)

    def ingest_file(self, os_filepath, virtual_filepath):
        filesize = os.path.getsize(os_filepath)
        created_at = int(time.time())
        chunk_metadata = []
        if os.path.exists(os_filepath):
            chunk_generator = self._chunked_coordinates(os_filepath)
            with concurrent.futures.ThreadPoolExecutor() as executor:
                mapper = executor.map(self._worker_encrypt, chunk_generator)
                for iv, ciphertext in mapper:
                    start_byte = self.vaultio.get_eof()
                    end_byte = start_byte + len(ciphertext)
                    self.vaultio.append_cargo(ciphertext)
                    chunk_metadata.append({
                        "iv" : iv,
                        "start" : start_byte,
                        "end" : end_byte        
                    })
                self.hashmap.add_file(virtual_filepath, chunk_metadata)
                meta_dict = {
                    "size_bytes" : filesize,
                    "created_at" : created_at,
                    "is_active" : True
                }
                self.vaultdb.inmo_file(virtual_filepath, meta_dict)
                self.sync_map_to_disc()
    
    #Category 3 : Extraction
    def _worker_decrypt(self, chunk_dict : dict):
        #This worker recieves one input dictionary we decrypt one chunk
        #{iv : 12bytes, start: int, end : int}
        self.aesgcm = AesGcm(self.hash)
        iv = chunk_dict.get('iv') 
        start = chunk_dict.get('start')
        end = chunk_dict.get('end')
        ciphertext = self.vaultio.read_cargo(start, end)
        plaintext = self.aesgcm.decrypt_cipherstream(iv, ciphertext)
        return plaintext

    def stream_file(self, virtual_path : str):
        chunk_array = self.hashmap.get_meta(virtual_path)
        if not chunk_array:
            raise FileNotFoundError(f"{virtual_path} not found in your map")
        else:
            with concurrent.futures.ThreadPoolExecutor() as executor:
                mapper = executor.map(self._worker_decrypt, chunk_array)

    def preview_file(self, virtual_path : str):
        self.ghostfile = GhostFileIO()
        ghost_path = self.ghostfile.create_ghost_path(virtual_path)
        for plaintext_chunk in self.stream_file(virtual_path):
            self.ghostfile.write_decrypted_chunk(ghost_path, plaintext_chunk)
        self.ghostfile.launch_file(ghost_path)




: 

In [ ]:
import os
import struct
import concurrent.futures

from modules.system import VaultIO
from modules.vaultmap import VaultMap
from modules.crypto_core import AesGcm 

class Defragmentation:
    def __init__(self, orchestrator):
        self.orchestrator = orchestrator
        self.vaultdb = orchestrator.vaultdb
        self.hashmap = orchestrator.hashmap
        self.vaultio = orchestrator.vaultio
        self.filepath = orchestrator.filepath
        self.tmp_path = self.filepath + ".tmp"


        self.vaultiot = VaultIO(self.tmp_path)
        self.hashmapt = VaultMap()
        self.is_running = False

    def _active_chunk_generator(self, active_files : list):
        #list of virtual paths that are active
        for virtual_paths in active_files:
            chunks = self.hashmap.get_meta(virtual_paths)
            for chunk in chunks:
                yield (
                    virtual_paths,
                    chunk['iv'],
                    chunk['start'],
                    chunk['end']
                )
    
    def _transfer_chunks(self, arguments : tuple):
        virtual_path, iv, start, end = arguments
        chunk_ct = self.vaultio.read_cargo(start, end)
        return (virtual_path, iv, chunk_ct)

    def sync_map_to_disc(self, new_map_dict: dict):
        local_aesgcm = AesGcm(self.orchestrator.hash)
        
       
        temp_hashmap = VaultMap()
        temp_hashmap.map = new_map_dict
        
        nmap = self.vaultdb.get_complete_meta()
        nmap_pt = temp_hashmap.restitch(nmap)
        
        
        nmap_iv, nmap_ciphertext = local_aesgcm.encrypt_cipherstream(nmap_pt)
        nmap_P = self.vaultiot.get_eof()
        nmap_L = len(nmap_iv) + len(nmap_ciphertext)
        self.vaultiot.append_map(nmap_iv, nmap_ciphertext) 

        identifier_pt = struct.pack('>QQ', nmap_P, nmap_L)
        iv_identifier, identifier_ct = local_aesgcm.encrypt_cipherstream(identifier_pt)

        with open(self.tmp_path, 'r+b') as tmp_file:
            tmp_file.seek(23)
            tmp_file.write(iv_identifier+ identifier_ct)
            os.fsync(tmp_file.fileno())

    def vacuum(self):
        if self.is_running:
            return False
        self.is_running = True
        try:
            with self.orchestrator.io_lock:
                
                active_files = self.orchestrator.vaultdb.get_active_files()
                new_map_dict = {vpath: [] for vpath in active_files}

                with open(self.tmp_path, 'wb') as wfile, open(self.orchestrator.filepath, 'rb') as rfile:
                    data = rfile.read(67)
                    wfile.write(data)


                with open(self.tmp_path, 'r+b') as rfile: 
                    with concurrent.futures.ThreadPoolExecutor() as executor:
                        mapper = executor.map(
                            self._transfer_chunks,
                            self._active_chunk_generator(active_files)
                            )
                        for virtual_path, iv, ciphertext in mapper:
                            start = self.vaultiot.get_eof()
                            end = start + len(ciphertext)
                            self.vaultiot.append_cargo(ciphertext)
                            new_map_dict[virtual_path].append({
                                "iv" : iv,
                                "start" : start,
                                "end" : end
                            })
                        os.fsync(rfile.fileno())

                if hasattr(self.vaultdb, 'purge_inactive_files'):
                    self.vaultdb.purge_inactive_files()
                self.sync_map_to_disc(new_map_dict)
                os.replace(self.tmp_path, self.filepath)
                self.hashmap.map = new_map_dict
                return True
        finally:
            self.is_running = False

In [ ]:
import oqs
from oqs.oqs import KeyEncapsulation





In [ ]:
import os
from oqs.oqs import KeyEncapsulation
from oqs.oqs import Signature
from modules.crypto_core import AesGcm

#Alice : 
example1 = KeyEncapsulation('ML-KEM-768')
public_key1 = example1.generate_keypair()
# Bob : 

iv = os.urandom(12)
example2 = KeyEncapsulation('ML-KEM-768')
public_key2 = example2.generate_keypair()
ciphertext1, shared_secret = example2.encap_secret(public_key1)
_, ciphertext2 = AesGcm(shared_secret).encrypt_cipherstream(b'Hello', iv)
#Alice
shared_secret2 = example1.decap_secret(ciphertext1)
plaintext = AesGcm(shared_secret2).decrypt_cipherstream(iv, ciphertext2)
print(plaintext)







b'Hello'


In [ ]:
from oqs.oqs import Signature

example = Signature('ML-DSA-87')
pub_= example.generate_keypair()
pvt = example.export_secret_key()

signature = example.sign(b'helloa')

example1 = Signature('ML-DSA-87')
verify = example1.verify(b'helloa', signature, pub_)
print(verify)

True


Help on function packb in module msgpack:

packb(o, **kwargs)
    Pack object `o` and return packed bytes
    
    See :class:`Packer` for options.



In [9]:
from oqs.oqs import Signature, KeyEncapsulation
from modules.crypto_core import AesGcm
import msgpack

#Alice
with KeyEncapsulation('ML-KEM-1024') as alice_kem:
    pub = alice_kem.generate_keypair()
    pvt = alice_kem.export_secret_key()

#Bob
with KeyEncapsulation('ML-KEM-1024') as bob_kem, Signature('ML-DSA-87') as bob_sign:
    kem_cipher, shared_secret = bob_kem.encap_secret(pub)
    local_aes = AesGcm(shared_secret)
    iv, aes_cipher = local_aes.encrypt_cipherstream(b'b1d2')
    stream = msgpack.packb((iv, aes_cipher), use_bin_type = True)
    sign_pub_key = bob_sign.generate_keypair()
    stream_signature = bob_sign.sign(stream)

#Alice :

with KeyEncapsulation('ML-KEM-1024', pvt) as alice_kem, Signature('ML-DSA-87') as alice_sign:
    
    verify1 = alice_sign.verify(stream, stream_signature, sign_pub_key)
    if verify1:
        shared_secret1 = alice_kem.decap_secret(kem_cipher)
        iv1, aes_cipher1 = msgpack.unpackb(stream)
        local_aes1 = AesGcm(shared_secret1)
        plaintext = local_aes1.decrypt_cipherstream(iv1, aes_cipher1)
        print(plaintext)



b'b1d2'


False

In [1]:
import socket

def get_local_ipv4():
    s = socket.socket(socket.AF_INET, socket.SOCK_DGRAM)
    try:
        s.connect(('8.8.8.8', 1))  #Google's DNS IP(any connection outside LAN) 
        ip = s.getsockname()[0]
    except:
        ip = 'ERROR'
    finally:
        s.close()
    return ip

get_local_ipv4()

    



'192.168.0.104'

In [17]:
get_local_ipv4()

'192.168.0.104'

In [69]:
from cryptography.hazmat.primitives.kdf.hkdf import HKDF as hkdf
from cryptography.hazmat.primitives.hashes import SHA3_256



class SASPass:
    def __init__ (self):
        pass
    
    def derive_pass(self, *public_keys : bytes):
        self.hash_object = hkdf(
            algorithm = SHA3_256(),
            length = 32,
            salt = None,
            info = b"_securevault._tcp.local.",
            backend = None
            )
        public_key_list = [p for p in public_keys]
        public_key_list.sort()
        combined_pass = b"".join(public_key_list)

        hash = self.hash_object.derive(combined_pass)
        pin = int.from_bytes(hash, byteorder='big')
        pin = pin % 1000000
        return f"{pin:06d}"

    def verify_pass(self, PIN : str  ,*public_keys : bytes):
        expected_pin = self.derive_pass(*public_keys)
        if expected_pin == PIN:
            return True
        else:
            return False



In [95]:
import os
a = os.urandom(32)
b = os.urandom(32)
pin = SASPass().derive_pass(a, b)
print(pin)




097432


In [98]:
check = SASPass().verify_pass('097432', b, a)
print(check)

True


In [ ]:
from oqs.oqs import Signature, KeyEncapsulation

class MLDSA:
    def __init__(self, secret_key: bytes = None):
        self.algorithm = 'ML-DSA-87'
        self.has_key = False
        
        if not secret_key:
            self.signature = Signature(self.algorithm)
        else:
            self.signature = Signature(self.algorithm, secret_key)
            self.has_key = True
    
    def generate_key(self):
        if not self.has_key:
            pub_key = self.signature.generate_keypair()
            _pvt_key = self.signature.export_secret_key()
            self.has_key = True
        else:
            _pvt_key = self.signature.export_secret_key()
            pub_key = _pvt_key[-2592:]

        return pub_key, _pvt_key
    
    def sign(self, message: bytes):
        if not self.has_key:
            raise ValueError("No secret key loaded. Generate or pass a key first.")
        return self.signature.sign(message)
    
    def verify(self, message: bytes, signature: bytes, public_key: bytes):
        with Signature('ML-DSA-87') as local_sign:
            try:
                return(local_sign.verify(message, signature, public_key))
            except Exception:
                return False
                
    def close(self):
        
        if hasattr(self, 'signature') and hasattr(self.signature, 'free'):
            self.signature.free()

    def __del__(self):
        self.close()

class MLKEM:
    def __init__(self, secret_key: bytes = None):
        self.algorithm = 'ML-KEM-1024'
        self.has_key = False
        self.pub_key = None
        self.pvt_key = None
        
        if secret_key:
            # Load the engine with the provided secret key
            self.keyencapsulation = KeyEncapsulation(self.algorithm, secret_key)
            self.has_key = True
        else:
            # Prepare an empty engine for generation later
            self.keyencapsulation = KeyEncapsulation(self.algorithm)
        
    def generate_key(self):
        """
        MANDATORY FIRST STEP:
        Populates self.pvt_key and self.pub_key whether starting fresh or from an existing key.
        """
        if not self.has_key:
            self.pub_key = self.keyencapsulation.generate_keypair()
            self.pvt_key = self.keyencapsulation.export_secret_key()
            self.has_key = True
        else:
            self.pvt_key = self.keyencapsulation.export_secret_key()
            self.pub_key = self.pvt_key[-1568:]
        
        return self.pvt_key, self.pub_key

    def encapsulate(self, peer_public_key: bytes):
        try:
            kem_cipher, shared_secret = self.keyencapsulation.encap_secret(peer_public_key)
            return kem_cipher, shared_secret
        except Exception:
            return False
    
    def decapsulate(self, kem_cipher: bytes):
        if not self.pvt_key:
            raise ValueError("State uninitialized. You must call generate_key() before decapsulating.")
            
        try:
            with KeyEncapsulation(self.algorithm, self.pvt_key) as decap:
                shared_secret = decap.decap_secret(kem_cipher)
                return shared_secret
        except Exception:
            return False

    def close(self):
        if hasattr(self, 'keyencapsulation') and hasattr(self.keyencapsulation, 'free'):
            self.keyencapsulation.free()
        self.pub_key = None
        self.pvt_key = None
    
    def __del__(self):
        self.close()


        
        

In [ ]:
import socket
import platform
import subprocess
import os

def get_device_identity():
    # 1. Base Device Name (Hostname)
    device_name = socket.gethostname()
    if device_name.endswith(".local"):
        device_name = device_name[:-6]
        
    system = platform.system()
    device_type = f"Unknown {system} Device"

    # 2. Attempt Android Detection (Java Native Interface)
    try:
        from jnius import autoclass
        Build = autoclass('android.os.Build')
        device_type = str(Build.MODEL)
        return device_name, device_type
    except ImportError:
        pass # Not Android, or pyjnius is not installed

    # 3. Attempt iOS Detection (Objective-C Bridge)
    try:
        from pyobjus import autoclass
        UIDevice = autoclass('UIDevice')
        device_type = str(UIDevice.currentDevice().name.UTF8String())
        return device_name, device_type
    except ImportError:
        pass # Not iOS, or pyobjus is not installed

    # 4. Standard Desktop / Server Detection
    try:
        if system == "Windows":
            cmd = 'powershell "Get-CimInstance -ClassName Win32_ComputerSystemProduct | Select-Object -ExpandProperty Name"'
            result = subprocess.check_output(cmd, shell=True, stderr=subprocess.DEVNULL)
            device_type = result.decode('utf-8').strip()
            
        elif system == "Darwin":  # macOS (since iOS was bypassed above)
            result = subprocess.check_output(["sysctl", "-n", "hw.model"], stderr=subprocess.DEVNULL)
            device_type = result.decode('utf-8').strip()
            
        elif system == "Linux":  # Standard Linux (since Android was bypassed above)
            filepath_dmi = "/sys/devices/virtual/dmi/id/product_name"
            filepath_dt = "/sys/firmware/devicetree/base/model"
            
            if os.path.exists(filepath_dmi):
                with open(filepath_dmi, "r") as f:
                    device_type = f.read().strip()
            elif os.path.exists(filepath_dt):
                with open(filepath_dt, "r") as f:
                    device_type = f.read().strip().strip('\x00')
    except Exception:
        # If any subprocess or file read fails, keep the fallback string
        pass

    return device_name, device_type

Device Name: VarunAsusV16
Device Model: ASUS Vivobook 16 V3607VU_V3607VU


: 

In [6]:
from hashlib import sha3_256
sha3_256(b'example').hexdigest()



'70983d692f648185febe6d6fa607630ae68649f7e6fc45b94680096c06e4fadb'

In [9]:
import os
os.path.splitext(os.path.basename("modules\daemons\syncledger.py"))[1]


'.py'

In [6]:
from cryptography.hazmat.primitives.asymmetric import mlkem
help(mlkem)

Help on module cryptography.hazmat.primitives.asymmetric.mlkem in cryptography.hazmat.primitives.asymmetric:

NAME
    cryptography.hazmat.primitives.asymmetric.mlkem

DESCRIPTION
    # This file is dual licensed under the terms of the Apache License, Version
    # 2.0, and the BSD License. See the LICENSE file in the root of this repository
    # for complete details.

CLASSES
    builtins.object
        MLKEM1024PrivateKey
        MLKEM1024PublicKey
        MLKEM768PrivateKey
        MLKEM768PublicKey
    
    class MLKEM1024PrivateKey(builtins.object)
     |  Methods defined here:
     |  
     |  __copy__(self) -> 'MLKEM1024PrivateKey'
     |      Returns a copy.
     |  
     |  __deepcopy__(self, memo: 'dict') -> 'MLKEM1024PrivateKey'
     |      Returns a deep copy.
     |  
     |  decapsulate(self, ciphertext: 'Buffer') -> 'bytes'
     |      Decapsulate: returns shared_secret.
     |  
     |  private_bytes(self, encoding: '_serialization.Encoding', format: '_serialization.Pr